Notebook to proceed with the following steps once the yolo model has been trained. Will run manually here sequences of the classify.py and testing.py scripts. 

Ideally, the runing of this section can be done by calling modes that run pipelines out of Charlie's code. Let's see how much of those automatisations can be used here. 
Here is his comment from the testing.py file:

Utility functions for training/validation pipeline.  
Includes: 

    - prepare: Prepare the images for segmentation
    - segment: Segment the images
    - run_detection: Run the YoloV5 detection
    - backwards_annotation: Generate labelme style annotations from the classifications
    - compare_detections_to_ground_truth: Match up labels and detections, compare them, and save the results
    - confusion_matrix: Summarize the results of the comparison

Load first all the configs and required packs:

In [ ]:

import os
import shutil
import yaml 
import argparse
import os.path as path
import scipy.cluster
import scipy.spatial
import json
import sys

import numpy as np
import pandas as pd
import scipy
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from tqdm import tqdm
import torch
import stat
import shutil
from datetime import datetime

from counting_boats.boat_utils.config import cfg
from counting_boats.boat_utils import image_cutting_support as ics
from counting_boats.boat_utils import heatmap as hm

import counting_boats.boat_utils.classifier
# cluster, process_clusters, read_classifications, pixel2latlong

# Add the project root to sys.path (adjust as needed)
sys.path.append(os.path.abspath("counting_waterholes"))


Then we can select manually which function we want to run. 
First, from the testing aoi tif file, we need to prepare the padded png image using the testing.prepare():

In [2]:
# Run preparation:
import counting_boats.boat_utils.testing


counting_boats.boat_utils.testing.prepare("counting_waterholes/testing", "config_test_GPU.yaml")

read the file path correctly
.\testing\pngs
.\testing\raw_images
Creating png for 20240101_mimal_test.tif
Doing gdal work...
Done with gdal work for 20240101_mimal_test.tif
New Width:  14976 New Height:  12064
Creating png for 20240204_mimal_test.tif
Doing gdal work...
Done with gdal work for 20240204_mimal_test.tif
New Width:  14976 New Height:  12064
Creating png for 20240324_mimal_test.tif
Doing gdal work...
Done with gdal work for 20240324_mimal_test.tif
New Width:  14976 New Height:  12064
Creating png for 20240415_mimal_test.tif
Doing gdal work...
Done with gdal work for 20240415_mimal_test.tif
New Width:  14976 New Height:  12064
Creating png for 20240604_mimal_test.tif
Doing gdal work...
Done with gdal work for 20240604_mimal_test.tif
New Width:  14976 New Height:  12064
Creating png for 20240704_mimal_test.tif
Doing gdal work...
Done with gdal work for 20240704_mimal_test.tif
New Width:  14976 New Height:  12064


Then I need to use the created png to label it with labelme. This will allow us to compare my annotation to the detection of the trained model i.e. test the model. 

Once the manual annotation is done, we can apply the segmentation used from the testing.segment():

In [ ]:
#run segmentation without spliting 80% of the images for validation!
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.segment(r"C:/Users/fossatia/OneDrive - Queensland University of Technology/FirstByte Waterholes WD/counting_waterholes/testing", "config_test_FirstByteDrive.yaml")

Cropping Image: .\testing\pngs\20240101_mimal_test.png
[12064 14976     3]
We will have:  15933  images maximum
0% of images without labels will be removed


Saving Segments: 100%|██████████| 15933/15933 [01:34<00:00, 169.26it/s]


Skipped 646 images
Empty 0 images
Cropping Image: .\testing\pngs\20240204_mimal_test.png


KeyboardInterrupt: 

Using those segmented labelled images, we can run the detection of waterholes using the trained model, and compare my label with the detection of the model. 

In [ ]:
#run detection on my testing 
# import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.run_detection("counting_waterholes/testing", "config_test_GPU.yaml")

Use the detection output of the model on my testing images to produce labelme style annotation using the backwards_annotation():  

In [ ]:
#run the annotation of the images using the detection of the model:
# import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.backwards_annotation_AF(run_folder= "counting_waterholes/testing", "config_test_GPU.yaml")

Then finally, compare the detected WH with my labeled WH:

In [ ]:
#comparison of my labels with the detected WH:
# import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.compare_detections_to_ground_truth(run_folder= "counting_waterholes/testing", "config_test_GPU.yaml")

Create the confusion matrix which summarises the results:

In [ ]:
#create the confusion matrix
# import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.confusion_matrix_AF(run_folder= "counting_waterholes/testing", "config_test_GPU.yaml")

Process the images by comparing the detections and labels for a single image

In [ ]:
#create the confusion matrix
# import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.process_image_AF(run_folder= "counting_waterholes/testing", "config_test_GPU.yaml")

Until here I should already have enough debugging to do before looking into more. But there are extra steps to compare image label detection and manual identification. 